In [1]:
import multiprocessing as mp
import concurrent.futures
from evaluate_model import evaluate_model

def run_eval_thread(create_params={}, train_params={}):
	context = mp.get_context("spawn")

	with concurrent.futures.ProcessPoolExecutor(max_workers=1, mp_context=context) as executor:
		future = executor.submit(evaluate_model, create_params, train_params)
		report, history, time_taken = future.result()
	
	return report, history, time_taken

In [2]:
import csv

def save_experiment_data(exp_name, data):
	keys = data[0].keys()

	with open(f"../data/experiment_{exp_name}.csv", "w") as f:
		writer = csv.writer(f)
		writer.writerow(keys)

		for row in data:
			writer.writerow(row.values())

In [ ]:
samples_avg = []

for i in range(9):
	validation_split = 0.99 - i * 0.01
	report, _, _ = run_eval_thread(train_params={ "validation_split": validation_split })
	samples_avg.append(report["samples avg"])

for i in range(9):
	validation_split = 0.9 - i * 0.1
	report, _, _ = run_eval_thread(train_params={ "validation_split": validation_split })
	samples_avg.append(report["samples avg"])

Epoch 1/10
128/128 [==============================] - 29s 217ms/step - loss: 0.2867 - accuracy: 0.2391 - val_loss: 0.1925 - val_accuracy: 0.3519
Epoch 2/10
128/128 [==============================] - 19s 149ms/step - loss: 0.1418 - accuracy: 0.5742 - val_loss: 0.1092 - val_accuracy: 0.6769
Epoch 3/10
128/128 [==============================] - 18s 140ms/step - loss: 0.0705 - accuracy: 0.7385 - val_loss: 0.0791 - val_accuracy: 0.6852
Epoch 4/10
128/128 [==============================] - 19s 148ms/step - loss: 0.0348 - accuracy: 0.7564 - val_loss: 0.0760 - val_accuracy: 0.6999
Epoch 5/10
128/128 [==============================] - 19s 147ms/step - loss: 0.0188 - accuracy: 0.7645 - val_loss: 0.0795 - val_accuracy: 0.6903
Epoch 6/10
319/319 [==============================] - 3s 9ms/step
Epoch 1/10
255/255 [==============================] - 31s 119ms/step - loss: 0.2199 - accuracy: 0.3810 - val_loss: 0.1104 - val_accuracy: 0.6508
Epoch 2/10
255/255 [==============================] - 25s 97ms/s

In [ ]:
save_experiment_data("data_split", samples_avg)

In [ ]:
report1, _, time_taken1 = run_eval_thread(create_params={"drop_data": 0.7})
report2, _, time_taken2 = run_eval_thread(create_params={"drop_data": 0})

time_taken1, time_taken2

Epoch 1/10
344/344 [==============================] - 21s 59ms/step - loss: 0.1909 - accuracy: 0.4602 - val_loss: 0.0845 - val_accuracy: 0.7010
Epoch 2/10
344/344 [==============================] - 20s 58ms/step - loss: 0.0610 - accuracy: 0.7254 - val_loss: 0.0573 - val_accuracy: 0.7083
Epoch 3/10
344/344 [==============================] - 20s 58ms/step - loss: 0.0309 - accuracy: 0.7535 - val_loss: 0.0571 - val_accuracy: 0.7181
Epoch 4/10
344/344 [==============================] - 20s 58ms/step - loss: 0.0177 - accuracy: 0.7623 - val_loss: 0.0659 - val_accuracy: 0.6985
Epoch 5/10
96/96 [==============================] - 1s 5ms/step
Epoch 1/10
1147/1147 [==============================] - 67s 58ms/step - loss: 0.1027 - accuracy: 0.6232 - val_loss: 0.0523 - val_accuracy: 0.7023
Epoch 2/10
1147/1147 [==============================] - 66s 58ms/step - loss: 0.0396 - accuracy: 0.7346 - val_loss: 0.0516 - val_accuracy: 0.7023
Epoch 3/10
1147/1147 [==============================] - 66s 58ms/ste

(101.63528323173523, 274.80494356155396)

In [6]:
report1["samples avg"]["precision"], report2["samples avg"]["precision"]

(0.8777378228179143, 0.8976930874370546)

In [6]:
_, history1, _ = run_eval_thread(
	create_params={"drop_data": 0.6, "dropout": 0},
	train_params={"stop_early": False, "epochs": 15}
)
_, history2, _ = run_eval_thread(
	create_params={"drop_data": 0.6, "dropout": 0.3},
	train_params={"stop_early": False, "epochs": 15}
)
_, history3, _ = run_eval_thread(
	create_params={"drop_data": 0.6, "dropout": 0.6},
	train_params={"stop_early": False, "epochs": 15}
)
loss_histories = [{
	"loss": h.history["loss"],
	"val_loss": h.history["val_loss"]
} for h in [history1, history2, history3]]

Epoch 1/15
459/459 [==============================] - 13s 28ms/step - loss: 0.1457 - accuracy: 0.5562 - val_loss: 0.0633 - val_accuracy: 0.7008
Epoch 2/15
459/459 [==============================] - 13s 28ms/step - loss: 0.0353 - accuracy: 0.7410 - val_loss: 0.0595 - val_accuracy: 0.6990
Epoch 3/15
459/459 [==============================] - 13s 28ms/step - loss: 0.0123 - accuracy: 0.7602 - val_loss: 0.0694 - val_accuracy: 0.6824
Epoch 4/15
459/459 [==============================] - 13s 28ms/step - loss: 0.0043 - accuracy: 0.7601 - val_loss: 0.0811 - val_accuracy: 0.6738
Epoch 5/15
459/459 [==============================] - 13s 28ms/step - loss: 0.0017 - accuracy: 0.7500 - val_loss: 0.0907 - val_accuracy: 0.6726
Epoch 6/15
459/459 [==============================] - 13s 28ms/step - loss: 8.8259e-04 - accuracy: 0.7432 - val_loss: 0.0979 - val_accuracy: 0.6609
Epoch 7/15
459/459 [==============================] - 14s 30ms/step - loss: 4.9356e-04 - accuracy: 0.7451 - val_loss: 0.1051 - val_a

In [10]:
data_rows = []

for i in range(15):
	data_rows.append({
		"epoch": i + 1,
		"low_loss": loss_histories[0]["loss"][i],
		"low_val_loss": loss_histories[0]["val_loss"][i],
		"med_loss": loss_histories[1]["loss"][i],
		"med_val_loss": loss_histories[1]["val_loss"][i],
		"high_loss": loss_histories[2]["loss"][i],
		"high_val_loss": loss_histories[2]["val_loss"][i]
	})

save_experiment_data("dropout", data_rows)

In [ ]:
samples_avg = []
counts = [50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000, 5000, 7500, 10000, 15000]
training_times = []

for count in counts:
	report, _, training_time = run_eval_thread(create_params={"max_features": count})
	samples_avg.append(report["samples avg"])
	training_times.append(training_time)

Epoch 1/10
1147/1147 [==============================] - 2s 2ms/step - loss: 0.1602 - accuracy: 0.4797 - val_loss: 0.1363 - val_accuracy: 0.5385
Epoch 2/10
1147/1147 [==============================] - 2s 2ms/step - loss: 0.1365 - accuracy: 0.5233 - val_loss: 0.1316 - val_accuracy: 0.5503
Epoch 3/10
1147/1147 [==============================] - 2s 2ms/step - loss: 0.1329 - accuracy: 0.5358 - val_loss: 0.1296 - val_accuracy: 0.5650
Epoch 4/10
1147/1147 [==============================] - 2s 2ms/step - loss: 0.1307 - accuracy: 0.5429 - val_loss: 0.1282 - val_accuracy: 0.5625
Epoch 5/10
1147/1147 [==============================] - 2s 2ms/step - loss: 0.1290 - accuracy: 0.5463 - val_loss: 0.1273 - val_accuracy: 0.5645
Epoch 6/10
1147/1147 [==============================] - 2s 2ms/step - loss: 0.1279 - accuracy: 0.5516 - val_loss: 0.1272 - val_accuracy: 0.5598
Epoch 7/10
1147/1147 [==============================] - 2s 1ms/step - loss: 0.1269 - accuracy: 0.5516 - val_loss: 0.1262 - val_accuracy:

In [ ]:
save_experiment_data("max_features", samples_avg)

In [6]:
training_times[0], training_times[-1]

(19.864384174346924, 145.80728936195374)